# 01 — Data Exploration

Verify access to all data sources and explore the study domain.

**Run this notebook first** to confirm everything works before building the pipeline.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Domain bounds (NW Russia)
LAT_MIN, LAT_MAX = 55.0, 70.0
LON_MIN, LON_MAX = 28.0, 55.0

## 1. ERA5 from WeatherBench 2

In [ ]:
# Load ERA5 — this is the ground truth you already have access to
era5 = xr.open_zarr(
    "gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr"
)

# Subset to domain
t2m_era5 = (
    era5["2m_temperature"]
    .sel(latitude=slice(LAT_MAX, LAT_MIN), longitude=slice(LON_MIN, LON_MAX), time="2022-01-15T12")
    .compute()
)

print(f"ERA5 shape: {t2m_era5.shape}")
print(f"Lat range: {float(t2m_era5.latitude.min()):.2f} to {float(t2m_era5.latitude.max()):.2f}")
print(f"Lon range: {float(t2m_era5.longitude.min()):.2f} to {float(t2m_era5.longitude.max()):.2f}")
print(
    f"T2m range: {float(t2m_era5.min()) - 273.15:.1f}°C to {float(t2m_era5.max()) - 273.15:.1f}°C"
)

In [ ]:
# Plot the domain
fig, ax = plt.subplots(1, 1, figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

t2m_celsius = t2m_era5 - 273.15
im = ax.pcolormesh(
    t2m_celsius.longitude,
    t2m_celsius.latitude,
    t2m_celsius,
    cmap="RdBu_r",
    vmin=-30,
    vmax=5,
    transform=ccrs.PlateCarree(),
)

ax.set_extent([LON_MIN - 2, LON_MAX + 2, LAT_MIN - 2, LAT_MAX + 2])
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.COASTLINE)
ax.gridlines(draw_labels=True, alpha=0.3)
ax.set_title("ERA5 2m Temperature — NW Russia (Jan 15, 2022 12Z)")
plt.colorbar(im, ax=ax, label="Temperature (°C)", shrink=0.7)
plt.tight_layout()
plt.show()

## 2. GEFS from dynamical.org (Zarr)

In [ ]:
# Load GEFS from dynamical.org — cloud-optimised Zarr
gefs = xr.open_zarr(
    "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr", chunks="auto"
)

print("GEFS variables:", list(gefs.data_vars))
print("GEFS dimensions:", dict(gefs.dims))
print()
print("Available coordinates:")
for c in gefs.coords:
    print(f"  {c}: {gefs[c].dims}, shape={gefs[c].shape}")

In [ ]:
# Subset GEFS to domain — this should be fast (lazy loading)
t2m_gefs = gefs["temperature_2m"].sel(
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX),
)

# Check the latest available init time
print(f"Latest init time: {t2m_gefs.init_time.values[-1]}")
print(f"Number of members: {t2m_gefs.sizes.get('member', 'N/A')}")
print(f"Domain grid points: lat={t2m_gefs.sizes['latitude']}, lon={t2m_gefs.sizes['longitude']}")

## 3. IMPROVER installation check

In [ ]:
import improver
import iris

print(f"IMPROVER version: {improver.__version__}")
print(f"Iris version: {iris.__version__}")

# Quick test: create a sample Iris cube and run threshold
import improver.cli as imprcli

# Load Iris sample data to test
sample = iris.load(iris.sample_data_path("E1_north_america.nc"))[0]
print(f"\nIris sample cube loaded: {sample.standard_name}")
print(f"Shape: {sample.shape}")

# Run IMPROVER threshold on sample data
output = imprcli.threshold.process(sample, threshold_values=[273.15, 280.0])
print(f"\nIMPROVER threshold output: {output.standard_name}")
print(f"Shape: {output.shape}")
print("\n✅ IMPROVER is working!")

## 4. Iris conversion test

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime
from src.data.iris_convert import xarray_to_iris_cube, validate_improver_compatibility

# Convert one ERA5 field to Iris cube
test_cube = xarray_to_iris_cube(
    da=t2m_era5,
    standard_name="air_temperature",
    units="K",
    model_id="era5",
    forecast_ref_time=datetime(2022, 1, 15, 12),
    forecast_period_hours=0,
)

print(test_cube)
print()

# Validate IMPROVER compatibility
issues = validate_improver_compatibility(test_cube)
if issues:
    print(f"⚠️  Issues: {issues}")
else:
    print("✅ Cube passes IMPROVER compatibility checks")


## Next steps

1. ✅ Verify ERA5 access
2. ✅ Verify GEFS access
3. ✅ Verify IMPROVER installation
4. ✅ Test Iris conversion
5. → Run `02_iris_conversion_test.ipynb` for full GEFS → Iris pipeline
6. → Run `03_improver_test_data.ipynb` to explore IMPROVER's expected formats